In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("glue", "mrpc")
raw_datasets

MRPC is a dataset for paraphrase detection. The task is to determine if two sentences are paraphrases of each other. The dataset consists of pairs of sentences, and the label indicates whether the sentences are paraphrases (1) or not (0).

In [ ]:
raw_train_dataset = raw_datasets["train"]

In [ ]:
raw_train_dataset

In [ ]:
raw_train_dataset[0]

In [ ]:
raw_train_dataset.features

## Pass sentence to tokenizer and get token ids

In [ ]:
from transformers import AutoTokenizer

In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
sentence_1 = raw_datasets["train"]["sentence1"]

In [ ]:
tokenized_sentences_1 = tokenizer(raw_datasets["train"]["sentence1"][0])

In [ ]:
tokenized_sentences_1

To evaluate wich sentence is a paraphrase or not we need to pass both sentences to the tokenizer.

In [ ]:
inputs = tokenizer(raw_datasets["train"]["sentence1"][0], raw_datasets["train"]["sentence2"][0], return_tensors="pt")


In [ ]:
inputs


token_type_ids are use to distinguish between the two sentences.

as mentioned in previous notebooks attention_mask is used to make length of the input sequance the same, because pytorch requires that all tensors be the same

In [ ]:
tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

In [ ]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

In [ ]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

In [ ]:
tokenized_datasets

In [ ]:
raw_datasets

## Dynamic padding

This technique is use to avoid unnecessary padding in each batch. Instead add padding for whole dataset as a whole, we adding padding for each batch separately, that padding be not in same length for a longest sentence in datasate but for a longest sentence in batch. This technique is use to save computational resources and make training faster.

Note. Dynamic padding is not applicable for TPU's.

In [ ]:
from transformers import DataCollatorWithPadding

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
samples = tokenized_datasets["train"][:8]

In [ ]:
samples

In [ ]:
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}


In [ ]:
samples

In [ ]:
[len(x) for x in samples["input_ids"]]

In [ ]:
batch = data_collator(samples)


In [ ]:
batch

In [ ]:
{k: v.shape for k, v in batch.items()}